# 질의분석(decompose+classify) O/X 비교 — 검색평가지표/latency (Evaluation_Dataset03 도전평가셋 74문항 + Evaluation_Dataset04 데이터셋_v5 180문항)

`experiment/eval01/run_bench.py`를 코랩에서 돌립니다. **두 데이터셋을 각각 따로** 채점합니다:
- **Evaluation_Dataset03.xlsx(`도전평가셋`)**: 콜로퀴얼체·복합질의·대명사·후속질문 등 질의분석이 실제로 필요하도록 일부러 어렵게 설계한 챌린지셋. route_type=RETRIEVE 74문항.
- **Evaluation_Dataset04.xlsx(`데이터셋_v5`)**: 넓은 범위의 일반 라우팅 골드셋. route_type=RETRIEVE 180문항.

문항마다 실제 파이프라인 순서 그대로(`decompose_query`로 질의분해 → 하위질문마다 `classify`로 라우팅/업무/의도 분류 → 검색)와 5가지 검색기법(dense v2 structured 단독 / hybrid 7:3 RRF·MinMax / hybrid 9:1 RRF·MinMax)을 돈 다음, 문항당 결과를 두 갈래로 같이 기록합니다.

- **latency**: 질의분석(`analysis_latency_sec`) / 검색(`search_latency_sec`) / 합계(`total_latency_sec`) — 질의분석 생략 시 검색만 걸리는 시간은 `search_latency_sec_raw`(=`total_latency_sec_raw`)
- **정확도(hit@3/recall@5/precision@5/f1@5/mrr@10/map@10)**: 질의분석을 거친 경우(`_qa` 접미사) vs 원문 질문을 그대로 검색한 경우(`_raw` 접미사, decompose/classify 없이 바로 검색)를 같이 뽑아서, 질의분석이 정확도에 실제로 얼마나 기여하는지(혹은 오버헤드만 되는지) 바로 비교할 수 있습니다.

두 데이터셋을 나눠 보는 이유: Dataset03(어려운 케이스만 모음)에서 질의분석 O/X 격차가 크게 나고 Dataset04(일반 케이스)에서는 격차가 작다면, "질의분석은 쉬운 질문에는 오버헤드일 뿐이고 어려운 질문에서만 이득"이라는 가설이 검증되는 것입니다. 합쳐서 채점하면 이 대비가 평균에 묻혀 안 보입니다.

**GPU가 필요 없습니다** — 이 실험은 로컬 딥러닝 모델(BGE-M3 등)을 전혀 쓰지 않고, dense는 이미 만들어둔 임베딩과의 코사인유사도(가벼운 CPU 연산) + HCX API 호출(네이버 서버에서 처리), BM25는 순수 파이썬(pynori)입니다. 런타임은 기본 CPU로 충분합니다 — 다만 로컬보다 상대적으로 여유 있는 CPU/네트워크를 쓰려고 코랩에서 돌립니다.

**중간 저장/재개**: 문항 하나 처리할 때마다 `experiment/eval01/results/cache/subq_cache_{데이터셋}.pkl`에 즉시 저장됩니다. 같은 런타임 안에서 API rate limit 등으로 중간에 죽었다가 4번 셀을 다시 실행하면 이미 처리된 문항은 건너뛰고 이어서 진행합니다(단, 코랩 런타임 자체가 끊겨서 `/content`가 초기화되면 캐시도 같이 사라집니다 — 그 경우엔 2번 셀부터 zip을 다시 올려서 재시작해야 합니다).

**준비물**: 로컬 `RAG_project3` 폴더 전체(코드 + `data/` 안의 jsonl 3종·`Evaluation_Dataset03.xlsx`·`Evaluation_Dataset04.xlsx`)를 zip으로 압축해두세요. `run_bench.py`가 새로 바뀌었으니 반드시 최신 상태로 다시 압축하세요.

## 1. 의존성 설치

Colab에 이미 있는 pandas/numpy/ipython은 건드리지 않고, 이 실험에 새로 필요한 것만 설치합니다.
- `openai`: HCX 클라이언트(OpenAI 호환 엔드포인트)
- `rank_bm25`, `openpyxl`: BM25 인덱스 / 평가 xlsx 로드
- `pynori`: Elasticsearch Nori 분석기의 순수 파이썬 포팅. PyPI에서 내려간 패키지라 GitHub 태그(0.1.2, 최신 마스터는 불필요한 의존성이 껴 있어서 이 태그를 씁니다)에서 직접 설치합니다.

In [ ]:
!pip install -q "openai>=1.68,<2" rank_bm25 openpyxl
!pip install -q "git+https://github.com/bage79/python-nori.git@0.1.2"

## 2. 프로젝트 업로드

로컬에서 `RAG_project3` 폴더를 zip으로 압축한 뒤 업로드하세요.

In [ ]:
from google.colab import files
import zipfile, os
from pathlib import Path

print("RAG_project3 폴더를 압축한 zip 파일을 업로드하세요...")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

extract_to = Path("/content/rag_project3_extracted")
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(extract_to)

project_root = None
for dirpath, dirnames, _ in os.walk(extract_to):
    if "core" in dirnames and "experiment" in dirnames:
        project_root = Path(dirpath)
        break

if project_root is None:
    print("experiment/ 폴더를 못 찾았어요 (구버전 zip일 수 있음) — 아래 트리를 확인하세요:")
    for root, dirs, _ in os.walk(extract_to):
        depth = str(root).replace(str(extract_to), "").count(os.sep)
        if depth <= 3:
            print("  " * depth + os.path.basename(root) + "/")
    raise RuntimeError("core/, experiment/ 폴더를 못 찾았어요. 최신 zip인지 확인해주세요.")

os.chdir(project_root)
print("프로젝트 루트로 이동 완료:", project_root)

for needed_data in [
    "data/Evaluation_Dataset03.xlsx",
    "data/Evaluation_Dataset04.xlsx",
]:
    p = project_root / needed_data
    print(needed_data, "->", "OK" if p.exists() else "!! 없음, zip을 다시 확인하세요")

## 2-1. dense v2 structured 임베딩 파일 확인

`rag_baseline/`은 `.gitignore` 대상이라 zip 도구에 따라(특히 IDE의 "프로젝트를 zip으로" 같은 VCS 인식 압축 기능을 쓰면) 통째로 빠질 수 있습니다. 그 안의 `results/experiments/chunk_embeddings_dense_structured.jsonl`(dense v2 structured가 쓰는 title+content 임베딩, 로컬 재계산 불가한 사전 캐시 파일, 약 5MB)이 없으면 이 셀에서 따로 업로드를 요청합니다. "없습니다"가 뜨면 로컬의 `RAG_project3/rag_baseline/results/experiments/chunk_embeddings_dense_structured.jsonl` 파일 하나만 선택해서 올려주세요.

In [ ]:
from pathlib import Path
from google.colab import files
import shutil

# 주의: 코랩에서는 core.config.WORK_ROOT가 프로젝트 폴더가 아니라
# /content/kdic_rag_baseline로 고정돼 있어서(core/config.py의 Colab 분기),
# 상대경로 "rag_baseline/..."가 아니라 core.config.EXPERIMENTS_ROOT를
# 그대로 가져와서 실제 코드가 찾는 경로에 맞춥니다.
from core.config import EXPERIMENTS_ROOT

needed = EXPERIMENTS_ROOT / "chunk_embeddings_dense_structured.jsonl"
if not needed.exists():
    print(f"{needed} 가 없습니다. 이 파일만 따로 업로드하세요:")
    uploaded2 = files.upload()
    needed.parent.mkdir(parents=True, exist_ok=True)
    fname = list(uploaded2.keys())[0]
    shutil.move(fname, needed)
    print("업로드 완료:", needed)
else:
    print("확인 완료:", needed, "존재함")

## 2-2. (선택) 이미 올린 세션이면 바뀐 파일만 패치

zip 전체(367MB)를 다시 올리면 오래 걸립니다. **이미 이 런타임에서 2번 셀로 zip을 한 번
업로드한 적이 있다면**(런타임을 재시작하지 않은 상태), 아래 셀로 바뀐 파일 몇 개만 골라
업로드해서 덮어쓸 수 있습니다. 런타임을 새로 시작했다면 이 셀은 건너뛰고 2번 셀부터
정상적으로 zip을 올리세요.

In [ ]:
from pathlib import Path
from google.colab import files
import shutil, os

if "project_root" not in globals():
    candidates = [p.parent for p in Path("/content/rag_project3_extracted").rglob("core")
                  if (p.parent / "experiment").exists()]
    if not candidates:
        raise RuntimeError("project_root를 못 찾았습니다 — 먼저 2번 셀로 zip을 한 번 업로드해야 합니다.")
    project_root = candidates[0]
    os.chdir(project_root)
    print("프로젝트 루트 재사용:", project_root)

# 새로 패치할 파일이 늘어나면 여기에 "업로드할 파일명": 목적지 경로만 추가하면 됩니다.
_DEST_MAP = {
    "run_bench.py": project_root / "experiment/eval01/run_bench.py",
    "Evaluation_Dataset05.xlsx": project_root / "data/Evaluation_Dataset05.xlsx",
}

print("바뀐 파일만 업로드하세요(여러 개 한 번에 선택 가능):", list(_DEST_MAP.keys()))
uploaded = files.upload()

for fname in uploaded:
    dest = _DEST_MAP.get(fname)
    if dest is None:
        print(f"  [경고] {fname}: 목적지를 몰라서 건너뜀 — _DEST_MAP에 추가하세요")
        continue
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(fname, dest)
    print(f"  {fname} -> {dest}")

## 3. HCX API 키 입력

In [ ]:
import getpass, os
os.environ["HCX_API_KEY"] = getpass.getpass("HCX_API_KEY 입력: ")

## 4. 본실행 (Dataset03 74문항 + Dataset04 180문항, 각각 x 5가지 검색기법)

BM25(Nori) 인덱스 빌드가 순수 파이썬이라 청크 427개 기준 처음 한 번 수 분 걸립니다. 이후 데이터셋별로 문항 진행 로그가 출력됩니다. 중간에 끊기면 이 셀을 다시 실행하면 캐시(`results/cache/subq_cache_*.pkl`)에서 이어서 진행합니다.

In [ ]:
!python -u experiment/eval01/run_bench.py

## 5. 결과 다운로드

`experiment/eval01/results/summary.csv`(데이터셋x방법별 요약, `dataset` 컬럼으로 구분)와 `detail_{데이터셋}_{방법}.csv`(문항별 상세, `_qa`/`_raw` 접미사로 질의분석 O/X 지표가 나란히 들어있음)를 zip으로 묶어 받습니다.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/eval01_results", "zip", "experiment/eval01/results")
files.download("/content/eval01_results.zip")

## 6. 로컬 세션에 알려줄 것

- 위 4번 셀 마지막에 출력되는 요약 표(두 데이터셋 합쳐서 10행 = 2데이터셋 x 5방법)를 그대로 로컬 세션에 붙여넣어 주세요.
- `dataset` 컬럼으로 `challenge_v3`(Dataset03)와 `routing_v5`(Dataset04)를 구분해서 볼 수 있습니다.
- 방법별로 `total_latency_sec`/`analysis_latency_sec`/`search_latency_sec`(질의분석 O)와 `total_latency_sec_raw`(질의분석 X), `hit@3_qa`/`recall@5_qa`(질의분석 O)와 `hit@3_raw`/`recall@5_raw`(질의분석 X)를 같이 확인할 수 있습니다.
- `analysis_latency_sec`은 같은 데이터셋 안에서는 5가지 방법이 모두 같은 값(decompose+classify는 검색기법과 무관하게 한 번만 실행되고 공유됨)인 게 정상입니다 — 방법별 차이는 `search_latency_sec`/`search_latency_sec_raw`에서만 납니다.
- 두 데이터셋 사이의 `_qa` vs `_raw` 격차 크기를 비교해서, "질의분석이 어려운 질문(Dataset03)에서 정말 이득인지, 쉬운 질문(Dataset04)에서는 오버헤드인지"를 판단해 주세요.